# Jackal MJX Env Test

This notebook defaults to `MUJOCO_GL=osmesa`, which renders correctly on this machine. If your setup has working EGL instead, you can switch it back before running the imports cell.

In [1]:
import os
os.environ.setdefault("MUJOCO_GL", "osmesa")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

import jax
import jax.numpy as jnp
import mediapy as media
import mujoco

import ss2r.benchmark_suites
from mujoco_playground import registry
from mujoco_playground._src import mjx_env


Warp DeprecationWarning: The namespace `warp.context` will soon be removed from the public API. It can still be accessed from `warp._src.context` but might be changed or removed without notice.
Warp DeprecationWarning: The symbol `warp.context.Module` will soon be removed from the public API. Use `warp.Module` instead.
Warp DeprecationWarning: The symbol `warp.context.get_module` will soon be removed from the public API. Use `warp.get_module` instead.


In [2]:
env_name = "JackalGoToGoalFlatTerrain"
cfg = registry.get_default_config(env_name)
env = registry.load(env_name, config=cfg)

(env_name, type(env).__name__, env.xml_path, env.action_size, env.dt, env.sim_dt)


('JackalGoToGoalFlatTerrain',
 'JackalGoToGoal',
 '/raid/users/tjrcjf410/sukchul/safe-learning/ss2r/benchmark_suites/mujoco_playground/jackal_joystick/assets/xmls/scene_mjx_flat_terrain.xml',
 2,
 0.02,
 0.004)

In [3]:
state = env.reset(jax.random.PRNGKey(0))

{
    "obs_shape": state.obs.shape,
    "goal": env.goal_position(state.data),
    "robot": env.robot_position(state.data),
    "goal_delta_body": env.goal_delta_body_frame(state.data),
    "hazards_shape": env.hazard_positions(state.data).shape,
    "obstacles_shape": env.obstacle_positions(state.data).shape,
}


{'obs_shape': (46,),
 'goal': Array([-0.21095037,  0.30728793,  0.35      ], dtype=float32),
 'robot': Array([1.289202  , 0.15282416, 0.0635    ], dtype=float32),
 'goal_delta_body': Array([ 1.1789314 , -0.94044495,  0.28649998], dtype=float32),
 'hazards_shape': (12, 3),
 'obstacles_shape': (10, 3)}

In [4]:
{
    "bodies": [mujoco.mj_id2name(env.mj_model, mujoco.mjtObj.mjOBJ_BODY, i) for i in range(env.mj_model.nbody)],
    "geoms": [mujoco.mj_id2name(env.mj_model, mujoco.mjtObj.mjOBJ_GEOM, i) for i in range(env.mj_model.ngeom)],
    "sensors": [mujoco.mj_id2name(env.mj_model, mujoco.mjtObj.mjOBJ_SENSOR, i) for i in range(env.mj_model.nsensor)],
}


{'bodies': ['world',
  'base_link',
  'chassis_link',
  'front_left_wheel_link',
  'front_right_wheel_link',
  'rear_left_wheel_link',
  'rear_right_wheel_link',
  'imu_link',
  'goal',
  'hazard_0',
  'hazard_1',
  'hazard_2',
  'hazard_3',
  'hazard_4',
  'hazard_5',
  'hazard_6',
  'hazard_7',
  'hazard_8',
  'hazard_9',
  'hazard_10',
  'hazard_11',
  'obstacle_0',
  'obstacle_1',
  'obstacle_2',
  'obstacle_3',
  'obstacle_4',
  'obstacle_5',
  'obstacle_6',
  'obstacle_7',
  'obstacle_8',
  'obstacle_9'],
 'geoms': ['floor',
  'chassis_collision',
  'chassis_visual',
  'front_fender',
  'rear_fender',
  'top_plate',
  'front_mount_geom',
  'rear_mount_geom',
  'front_left_wheel',
  'front_right_wheel',
  'rear_left_wheel',
  'rear_right_wheel',
  'goal',
  'hazard_0',
  'hazard_1',
  'hazard_2',
  'hazard_3',
  'hazard_4',
  'hazard_5',
  'hazard_6',
  'hazard_7',
  'hazard_8',
  'hazard_9',
  'hazard_10',
  'hazard_11',
  'obstacle_0_visual',
  'obstacle_0_collider_x',
  'obstac

In [5]:
def policy(state):
    goal = env.goal_delta_body_frame(state.data)
    dist = jnp.linalg.norm(goal[:2])
    heading = jnp.arctan2(goal[1], goal[0])
    forward = jnp.clip(0.7 * dist, 0.0, 1.0)
    turn = jnp.clip(1.5 * heading, -1.0, 1.0)
    return jnp.clip(jnp.array([forward - turn, forward + turn]), -1.0, 1.0)

seconds = 3
rollout = [state]
actions = []
rewards = []
costs = []
goals = []
jit_step = jax.jit(env.step)
for _ in range(int(seconds / env.dt)):
    action = policy(state)
    state = jit_step(state, action)
    rollout.append(state)
    actions.append(action)
    rewards.append(float(state.reward))
    costs.append(float(state.info["cost"]))
    goals.append(float(jnp.linalg.norm(env.goal_delta(state.data)[:2])))


{
    "steps": len(rollout) - 1,
    "final_reward": rewards[-1],
    "final_cost": costs[-1],
    "cumulative_cost": sum(costs),
    "final_goal_distance": goals[-1],
    "goal_reached": float(state.info["goal_reached"]),
    "done": float(state.done),
}


{'steps': 150,
 'final_reward': 0.9994078874588013,
 'final_cost': 0.0,
 'cumulative_cost': 0.0,
 'final_goal_distance': 0.19843623042106628,
 'goal_reached': 1.0,
 'done': 0.0}

In [6]:
frames = env.render(rollout)
media.show_video(frames, fps=1.0 / env.dt)


100%|██████████| 151/151 [00:06<00:00, 23.92it/s]


This browser does not support the video tag.


In [7]:
active_contacts = []
for geom_pair, dist in zip(state.data.contact.geom, state.data.contact.dist):
    if float(dist) < 0:
        g1 = mujoco.mj_id2name(env.mj_model, mujoco.mjtObj.mjOBJ_GEOM, int(geom_pair[0]))
        g2 = mujoco.mj_id2name(env.mj_model, mujoco.mjtObj.mjOBJ_GEOM, int(geom_pair[1]))
        active_contacts.append((g1, g2, float(dist)))

{
    "accelerometer": mjx_env.get_sensor_data(env.mj_model, state.data, "imu_accel"),
    "goal_delta_body": env.goal_delta_body_frame(state.data),
    "active_contacts": active_contacts[:20],
}


/tmp/ipykernel_337160/1359033125.py:2: DeprecationWarning: Accessing `contact` directly from `Data` is deprecated. Access it via `data._impl.contact` instead.
  for geom_pair, dist in zip(state.data.contact.geom, state.data.contact.dist):


{'accelerometer': Array([-3.17866   , -0.84065497, -0.66369295], dtype=float32),
 'goal_delta_body': Array([ 0.17101538, -0.0707171 ,  0.24334908], dtype=float32),
 'active_contacts': []}